# Building an Investment Advisor AI

By the end of this notebook you'll have a 4-agent system that **fetches financial data**, **gathers latest news**, **analyzes both**, and **produces a structured investment recommendation** with guardrails — all running in parallel where possible.

We build it one concept at a time.

| Step | Concept | What It Unlocks |
|------|---------|----------------|
| 1 | **Custom Tools** | Agents that pull real financial data from Yahoo Finance |
| 2 | **First Agent** | A data researcher that uses your tools |
| 3 | **Second Agent + Web Search** | A news researcher with internet access |
| 4 | **Context Passing** | Analysis agent receives output from other agents |
| 5 | **Multi-Crew + Parallel** | Run data + news gathering simultaneously |
| 6 | **Structured Output** | Get a typed `InvestmentRecommendation` instead of raw text |
| 7 | **Guardrails + Full Pipeline** | Validate the recommendation before accepting it |

In [27]:
import json
import os
import time

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

---

## Step 1: Custom Tools

In the intermediate project, we used pre-built tools like `FileReadTool` and `EXASearchTool`. Now we'll build our own.

### Why `@tool`?

CrewAI agents can only call functions that are wrapped with the `@tool` decorator. It does two things:

1. **Registers** the function so the agent knows it exists and can call it
2. **Extracts** the function's name, docstring, and parameter types so the agent knows *what* the tool does and *how* to call it

Without `@tool`, the function is just a regular Python function — invisible to the agent.

We'll create three tools using the `yfinance` library to pull real financial data.

In [28]:
import yfinance as yf
from crewai.tools import tool
from curl_cffi import requests

session = requests.Session(impersonate="chrome")

In [29]:
@tool("Get current stock price")
def get_current_stock_price(symbol: str) -> str:
    """Use this function to get the current stock price for a given symbol.

    Args:
        symbol (str): The stock symbol.

    Returns:
        str: The current stock price or error message.
    """
    try:
        time.sleep(0.5)
        stock = yf.Ticker(symbol, session=session)
        current_price = stock.info.get(
            "regularMarketPrice", stock.info.get("currentPrice")
        )
        return (
            f"{current_price:.2f}"
            if current_price
            else f"Could not fetch current price for {symbol}"
        )
    except Exception as e:
        return f"Error fetching current price for {symbol}: {e}"

In [30]:
@tool
def get_company_info(symbol: str):
    """Use this function to get company information and current financial snapshot for a given stock symbol.

    Args:
        symbol (str): The stock symbol.

    Returns:
        JSON containing company profile and current financial snapshot.
    """
    try:
        company_info_full = yf.Ticker(symbol, session=session).info
        if company_info_full is None:
            return f"Could not fetch company info for {symbol}"

        company_info_cleaned = {
            "Name": company_info_full.get("shortName"),
            "Symbol": company_info_full.get("symbol"),
            "Current Stock Price": f"{company_info_full.get('regularMarketPrice', company_info_full.get('currentPrice'))} {company_info_full.get('currency', 'USD')}",
            "Market Cap": f"{company_info_full.get('marketCap', company_info_full.get('enterpriseValue'))} {company_info_full.get('currency', 'USD')}",
            "Sector": company_info_full.get("sector"),
            "Industry": company_info_full.get("industry"),
            "Country": company_info_full.get("country"),
            "EPS": company_info_full.get("trailingEps"),
            "P/E Ratio": company_info_full.get("trailingPE"),
            "52 Week Low": company_info_full.get("fiftyTwoWeekLow"),
            "52 Week High": company_info_full.get("fiftyTwoWeekHigh"),
            "Revenue Growth": company_info_full.get("revenueGrowth"),
            "Gross Margins": company_info_full.get("grossMargins"),
            "EBITDA": company_info_full.get("ebitda"),
        }
        return json.dumps(company_info_cleaned)
    except Exception as e:
        return f"Error fetching company profile for {symbol}: {e}"

In [31]:
@tool
def get_income_statements(symbol: str):
    """Use this function to get income statements for a given stock symbol.

    Args:
        symbol (str): The stock symbol.

    Returns:
        JSON containing income statements or an empty dictionary.
    """
    try:
        stock = yf.Ticker(symbol, session=session)
        financials = stock.financials
        return financials.to_json(orient="index")
    except Exception as e:
        return f"Error fetching income statements for {symbol}: {e}"

In [32]:
print("Tools created:")
print(f"  - {get_current_stock_price.name}")
print(f"  - {get_company_info.name}")
print(f"  - {get_income_statements.name}")

Tools created:
  - Get current stock price
  - get_company_info
  - get_income_statements


Let's see what `@tool` actually registers — this is what the agent sees when it decides which tool to call.

In [33]:
print(get_current_stock_price.description)
print()
print("Args schema:", get_current_stock_price.args_schema.model_json_schema())

Tool Name: Get current stock price
Tool Arguments: {'symbol': {'description': None, 'type': 'str'}}
Tool Description: Use this function to get the current stock price for a given symbol.

    Args:
        symbol (str): The stock symbol.

    Returns:
        str: The current stock price or error message.
    

Args schema: {'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'Getcurrentstockprice', 'type': 'object'}


In [34]:
print(get_company_info.description)
print()
print("Args schema:", get_company_info.args_schema.model_json_schema())

Tool Name: get_company_info
Tool Arguments: {'symbol': {'description': None, 'type': 'str'}}
Tool Description: Use this function to get company information and current financial snapshot for a given stock symbol.

    Args:
        symbol (str): The stock symbol.

    Returns:
        JSON containing company profile and current financial snapshot.
    

Args schema: {'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'Get_Company_Info', 'type': 'object'}


In [35]:
print(get_income_statements.description)
print()
print("Args schema:", get_income_statements.args_schema.model_json_schema())

Tool Name: get_income_statements
Tool Arguments: {'symbol': {'description': None, 'type': 'str'}}
Tool Description: Use this function to get income statements for a given stock symbol.

    Args:
        symbol (str): The stock symbol.

    Returns:
        JSON containing income statements or an empty dictionary.
    

Args schema: {'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'Get_Income_Statements', 'type': 'object'}


The agent reads the **Tool Name**, **Tool Arguments** (with types), and **Tool Description** (from the docstring) to decide when and how to call the function. Let's also verify the tool works.

In [36]:
print(get_current_stock_price.run("AAPL"))

Using Tool: Get current stock price
272.07


---

## Step 2: First Agent — Data Researcher

This agent uses our custom tools to pull financial data for any stock. We give it `get_company_info` and `get_income_statements` so it can gather fundamental data.

Let's run it once to see our custom tools in action.

In [ ]:
data_explorer = Agent(
    role="Data Researcher",
    goal="Gather and provide financial data and company information about a stock",
    llm=llm,
    verbose=True,
    backstory=(
        "You are an expert researcher, who can gather detailed information about a company or stock. "
        'When using tools, use the stock symbol and add a suffix \".NS\" to it. '
        'Try with and without the suffix and see what works.'
    ),
    tools=[get_company_info, get_income_statements],
    max_iter=5,
    max_rpm=12,
    max_execution_time=450,
    respect_context_window=True,
)

financials_task = Task(
    description="Get financial data like income statements and other fundamental ratios for stock: {stock}. "
    "Use the year 2026 as the current year.",
    expected_output="Detailed information from income statement, key ratios for {stock}. "
    "Indicate also about current financial status and trend over the period.",
    agent=data_explorer,
)

crew = Crew(
    agents=[data_explorer],
    tasks=[financials_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff(inputs={"stock": "AAPL"})
print(result.raw[:500])

The agent called our custom tools to pull real financial data. But financial data alone isn't enough — what's happening in the news can move a stock more than its earnings report.

---

## Step 3: Second Agent + Web Search

This agent uses `EXASearchTool` to search the internet for the latest news and developments about a company. It works the same way as the Data Researcher — different role, different tool.

In [ ]:
from crewai_tools import EXASearchTool

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY")
exa_search_tool = EXASearchTool()

In [ ]:
news_info_explorer = Agent(
    role="News and Info Researcher",
    goal="Gather and provide the latest news and information about a company from the internet",
    llm=llm,
    verbose=True,
    backstory="You are an expert researcher, who can gather detailed information about a company.",
    tools=[exa_search_tool],
    max_iter=5,
    max_rpm=15,
    max_execution_time=600,
    respect_context_window=True,
)

print(f"Agent created: {news_info_explorer.role}")
print(f"Tool: {exa_search_tool.name}")

Now we have two agents: one pulls financial data, the other gathers news. But they're working independently. The real power comes when we combine their outputs.

---

## Step 4: Context Passing

The `context` parameter on a Task tells CrewAI: *"Before this agent starts, give it the output from these other tasks."*

Our analyst agent doesn't need any tools — it receives the financial data AND the news automatically and consolidates them into a comprehensive analysis.

We'll define the analyst here, but won't run it yet — in the next step we'll wire everything together with parallel execution.

In [ ]:
analyst = Agent(
    role="Data Analyst",
    goal="Consolidate financial data, stock information, and provide a summary",
    llm=llm,
    verbose=True,
    backstory=(
        "You are an expert in analyzing financial data, stock/company-related current information, and "
        "making a comprehensive analysis."
    ),
    max_iter=4,
    max_rpm=10,
    max_execution_time=300,
    respect_context_window=True,
)

print(f"Agent created: {analyst.role}")
print("This agent has no tools — it synthesizes findings from the other two agents via context.")

---

## Step 5: Multi-Crew + Parallel Execution

The data agent and news agent don't depend on each other — so why run them one after the other? We split them into **separate crews** and run both at the same time using Python's `ThreadPoolExecutor`.

The analysis crew waits for both to finish, then runs sequentially. We'll define the parallel execution pattern here and run the complete system in the full pipeline at the end.

```
┌─────────────────┐     ┌─────────────────┐
│  financial_crew  │     │    news_crew     │
│  (data agent)    │     │  (news agent)    │
└────────┬────────┘     └────────┬────────┘
         │    runs in parallel    │
         └──────────┬─────────────┘
                    ▼
         ┌─────────────────────┐
         │   analysis_crew     │
         │ (analyst + expert)  │
         └─────────────────────┘
```

In [ ]:
fin_expert = Agent(
    role="Financial Expert",
    goal="Considering financial analysis of a stock, make investment recommendations",
    backstory=(
        "You are an expert financial advisor who can provide investment recommendations. "
        "Consider the financial analysis, current information about the company, current stock price, "
        "and make recommendations about whether to buy/hold/sell a stock along with reasons. "
        'When using tools, try with and without the suffix \".NS\" to the stock symbol and see what works.'
    ),
    llm=llm,
    verbose=True,
    tools=[get_current_stock_price],
    max_iter=5,
    max_rpm=8,
    max_execution_time=360,
    respect_context_window=True,
)

print("4 agents ready:")
print(f"  1. {data_explorer.role}")
print(f"  2. {news_info_explorer.role}")
print(f"  3. {analyst.role}")
print(f"  4. {fin_expert.role}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def run_crew_task(crew, inputs, task_name):
    print(f"Starting {task_name}...")
    result = crew.kickoff(inputs=inputs)
    print(f"Completed {task_name}")
    return result


print("Parallel execution pattern:")
print("  1. financial_crew and news_crew run simultaneously via ThreadPoolExecutor")
print("  2. analysis_crew waits for both, then runs sequentially")
print("\nWe'll use this in the full pipeline below.")

Right now the pipeline would produce raw text. To use this in a real application, you'd need to parse the text to extract the recommendation, confidence, price targets, etc. That's fragile and error-prone.

---

## Step 6: Structured Output

Instead of parsing raw text, we define a Pydantic model that describes exactly what the output should look like. CrewAI forces the agent to fill in every field.

| Without `output_pydantic` | With `output_pydantic` |
|--------------------------|------------------------|
| Raw text you have to parse | A typed Python object |
| `"I recommend a BUY..."` | `rec.action == "BUY"` |
| Fragile string matching | `rec.confidence == 0.85` |

We'll define the model here and use it in the full pipeline in the next step.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class InvestmentRecommendation(BaseModel):
    action: Literal["BUY", "HOLD", "SELL"] = Field(description="Investment action")
    confidence: float = Field(description="Confidence score 0.0 to 1.0")
    target_price: float = Field(description="12-month target price")
    current_price: float = Field(description="Current stock price")
    reasons: list[str] = Field(description="Key reasons for recommendation")
    risks: list[str] = Field(description="Key risks to consider")


print("InvestmentRecommendation fields:")
for name, field in InvestmentRecommendation.model_fields.items():
    print(f"  {name}: {field.annotation} — {field.description}")

Adding `output_pydantic=InvestmentRecommendation` to a task will force the agent to produce output matching this exact schema. But what if the agent returns a confidence of 5.0 or only gives one reason? We need validation — that's what guardrails are for.

---

## Step 7: Guardrails + Full Pipeline

A **guardrail** is a validation function that checks the agent's output before accepting it. If it fails, CrewAI sends the feedback back to the agent to try again.

The function must return `(True, result)` if valid or `(False, "error message")` if not.

Below is the complete system: all 4 agents, 3 crews, parallel execution, structured output, guardrails, and file output — everything in one cell so you can see the complete system.

In [ ]:
from typing import Any, Tuple

from crewai.tasks.task_output import TaskOutput


def validate_recommendation(result: TaskOutput) -> Tuple[bool, Any]:
    rec = result.pydantic
    if not rec or rec.confidence < 0 or rec.confidence > 1:
        return (False, "Confidence must be between 0.0 and 1.0")
    if len(rec.reasons) < 2:
        return (False, "Must provide at least 2 reasons for the recommendation")
    if len(rec.risks) < 1:
        return (False, "Must provide at least 1 risk")
    return (True, rec)


print("Guardrail ready: validate_recommendation")
print("  - Checks confidence is between 0.0 and 1.0")
print("  - Requires at least 2 reasons")
print("  - Requires at least 1 risk")

In [ ]:
os.makedirs("task_outputs", exist_ok=True)

data_explorer = Agent(
    role="Data Researcher",
    goal="Gather and provide financial data and company information about a stock",
    llm=llm,
    verbose=True,
    backstory=(
        "You are an expert researcher, who can gather detailed information about a company or stock. "
        'When using tools, use the stock symbol and add a suffix \".NS\" to it. '
        'Try with and without the suffix and see what works.'
    ),
    tools=[get_company_info, get_income_statements],
    max_iter=5,
    max_rpm=12,
    max_execution_time=450,
    respect_context_window=True,
)

news_info_explorer = Agent(
    role="News and Info Researcher",
    goal="Gather and provide the latest news and information about a company from the internet",
    llm=llm,
    verbose=True,
    backstory="You are an expert researcher, who can gather detailed information about a company.",
    tools=[exa_search_tool],
    max_iter=5,
    max_rpm=15,
    max_execution_time=600,
    respect_context_window=True,
)

analyst = Agent(
    role="Data Analyst",
    goal="Consolidate financial data, stock information, and provide a summary",
    llm=llm,
    verbose=True,
    backstory=(
        "You are an expert in analyzing financial data, stock/company-related current information, and "
        "making a comprehensive analysis."
    ),
    max_iter=4,
    max_rpm=10,
    max_execution_time=300,
    respect_context_window=True,
)

fin_expert = Agent(
    role="Financial Expert",
    goal="Considering financial analysis of a stock, make investment recommendations",
    backstory=(
        "You are an expert financial advisor who can provide investment recommendations. "
        "Consider the financial analysis, current information about the company, current stock price, "
        "and make recommendations about whether to buy/hold/sell a stock along with reasons. "
        'When using tools, try with and without the suffix \".NS\" to the stock symbol and see what works.'
    ),
    llm=llm,
    verbose=True,
    tools=[get_current_stock_price],
    max_iter=5,
    max_rpm=8,
    max_execution_time=360,
    respect_context_window=True,
)

financials_task = Task(
    description="Get financial data like income statements and other fundamental ratios for stock: {stock}. "
    "Use the year 2026 as the current year.",
    expected_output="Detailed information from income statement, key ratios for {stock}. "
    "Indicate also about current financial status and trend over the period.",
    agent=data_explorer,
)

news_task = Task(
    description="Get latest news and business information about company: {stock}. "
    "Use the year 2026 as the current year.",
    expected_output="Latest news and business information about the company. Provide a summary also.",
    agent=news_info_explorer,
)

analysis_task = Task(
    description="Make thorough analysis based on given financial data and latest news of a stock.",
    expected_output="Comprehensive analysis of a stock outlining financial health, stock valuation, risks, and news.",
    agent=analyst,
    context=[financials_task, news_task],
    output_file="task_outputs/financial_analysis.md",
)

advise_task = Task(
    description="Make a recommendation about investing in a stock, based on analysis provided and current stock price. "
    "Explain the reasons.",
    expected_output="A structured investment recommendation with action, confidence, target price, reasons, and risks.",
    agent=fin_expert,
    context=[analysis_task],
    output_pydantic=InvestmentRecommendation,
    guardrail=validate_recommendation,
    output_file="task_outputs/investment_recommendation.md",
)

financial_crew = Crew(
    agents=[data_explorer],
    tasks=[financials_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=15,
)

news_crew = Crew(
    agents=[news_info_explorer],
    tasks=[news_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=15,
)

analysis_crew = Crew(
    agents=[analyst, fin_expert],
    tasks=[analysis_task, advise_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=15,
)

stock_input = {"stock": "AAPL"}

parallel_start = time.time()
with ThreadPoolExecutor(max_workers=2) as executor:
    financial_future = executor.submit(
        run_crew_task, financial_crew, stock_input, "Financial Data Gathering"
    )
    news_future = executor.submit(
        run_crew_task, news_crew, stock_input, "News Gathering"
    )
    financial_result = financial_future.result()
    news_result = news_future.result()
parallel_time = time.time() - parallel_start

analysis_start = time.time()
final_result = analysis_crew.kickoff(inputs=stock_input)
analysis_time = time.time() - analysis_start

total_time = parallel_time + analysis_time
print(f"\nPhase 1 (parallel): {parallel_time:.2f}s")
print(f"Phase 2 (sequential): {analysis_time:.2f}s")
print(f"Total: {total_time:.2f}s")

In [ ]:
rec = final_result.pydantic

print(f"Action:     {rec.action}")
print(f"Confidence: {rec.confidence}")
print(f"Target:     ${rec.target_price}")
print(f"Current:    ${rec.current_price}")
print(f"\nReasons:")
for r in rec.reasons:
    print(f"  - {r}")
print(f"\nRisks:")
for r in rec.risks:
    print(f"  - {r}")

print(f"\nRaw output saved to: task_outputs/investment_recommendation.md")
print(f"Analysis saved to:   task_outputs/financial_analysis.md")

---

## Recap

| Step | You Learned | CrewAI API |
|------|------------|------------|
| 1 | Build custom tools | `@tool` decorator with `yfinance` |
| 2 | Create an agent with tools | `Agent(tools=[...])` |
| 3 | Add web search capability | `EXASearchTool()` |
| 4 | Chain task outputs | `context=[upstream_task]` |
| 5 | Multi-crew + parallel execution | `ThreadPoolExecutor`, separate `Crew` per concern |
| 6 | Get typed output | `output_pydantic=Model` |
| 7 | Validate + full pipeline | `guardrail=validate_fn`, `output_file` |

**Next** → Open `02_production_features.ipynb` to add async multi-stock screening and intelligent routing with Flows.